# CUAD Dataset Explorer
Explore the CUAD dataset — browse contracts, inspect QA pairs, and check clause distributions.

In [ ]:
from datasets import load_dataset

ds = load_dataset('kenlevine/CUAD', split='train', verification_mode='no_checks')
contracts = ds[0]['data']
print(f'Total contracts: {len(contracts)}')
print(f'QA pairs per contract: {len(contracts[0]["paragraphs"][0]["qas"])}')
print(f'Total QA pairs: {len(contracts) * len(contracts[0]["paragraphs"][0]["qas"])}')

## Browse a Contract
Change `CONTRACT_INDEX` to any number from 0 to 509 to pick a different contract.

In [ ]:
CONTRACT_INDEX = 8  # change this (0–509)

contract = contracts[CONTRACT_INDEX]
context = contract['paragraphs'][0]['context']
qas = contract['paragraphs'][0]['qas']

print(f'Contract title: {contract["title"]}')
print(f'Contract length: {len(context)} characters')
print()
print('--- Contract text (first 800 chars) ---')
print(context[:800])

## Show All 41 QA Pairs for the Selected Contract

In [ ]:
for i, qa in enumerate(qas):
    has_answer = bool(qa['answers'])
    clause_type = qa['question'].split('"')[1] if '"' in qa['question'] else qa['question']
    answer = qa['answers'][0]['text'][:150] if has_answer else '(no clause)'
    status = '✓' if has_answer else '✗'
    print(f'[{i+1:02d}] {status} {clause_type}')
    print(f'      {answer}')
    print()

## Inspect a Specific QA Pair with Surrounding Context
Change `QA_INDEX` to any number from 0 to 40.

In [ ]:
QA_INDEX = 9  # change this (0–40)
CONTEXT_WINDOW = 200  # chars before/after the answer span

qa = qas[QA_INDEX]
print('QUESTION:')
print(qa['question'])
print()

if qa['answers']:
    ans = qa['answers'][0]
    start = ans['answer_start']
    end = start + len(ans['text'])
    print('ANSWER:')
    print(ans['text'])
    print(f'\nchar position: {start} → {end}')
    print()
    print('SURROUNDING CONTEXT:')
    before = context[max(0, start - CONTEXT_WINDOW):start]
    after = context[end:end + CONTEXT_WINDOW]
    print(f'...{before}[[ {ans["text"]} ]]{after}...')
else:
    print('ANSWER: (no clause found in this contract)')

## Clause Coverage Across All Contracts
How often does each clause type appear?

In [ ]:
from collections import defaultdict

clause_counts = defaultdict(int)

for c in contracts:
    for qa in c['paragraphs'][0]['qas']:
        clause_type = qa['question'].split('"')[1] if '"' in qa['question'] else qa['question'][:40]
        if qa['answers']:
            clause_counts[clause_type] += 1

print(f'{'Clause Type':<45} {'Count':>6}  {'% of contracts':>14}')
print('-' * 70)
for clause, count in sorted(clause_counts.items(), key=lambda x: -x[1]):
    pct = count / len(contracts) * 100
    print(f'{clause:<45} {count:>6}  {pct:>13.1f}%')

## Search for Contracts by Keyword
Find all contracts whose title contains a keyword.

In [ ]:
KEYWORD = 'license'  # change this

matches = [
    (i, c['title'])
    for i, c in enumerate(contracts)
    if KEYWORD.lower() in c['title'].lower()
]

print(f'Found {len(matches)} contracts matching "{KEYWORD}":')
for idx, title in matches[:20]:
    print(f'  [{idx:03d}] {title}')